In [1]:
# =============================================================================
# DESI DR2 BAO + CMB + DES-SN5YR (Dovekie) Joint Analysis
# =============================================================================
#
# CAMB predictions -> Gaussian likelihoods -> Nautilus nested sampling
# -> posterior covariance -> pivot redshift -> corner plot
#
# Three probes:
#   1. DESI DR2 BAO (distance ratios)
#   2. CMB compressed likelihood (R, La, Ωb h²)
#   3. DES-SN5YR supernovae (Dovekie) with analytic M marginalization
#
# =============================================================================

import numpy as np
import numpy.linalg as la
from dataclasses import dataclass
from typing import Tuple, Optional
import warnings
import os

# plotting
import matplotlib.pyplot as plt
import corner

# sampler
from nautilus import Sampler

# CAMB
import camb


C_LIGHT = 299792.458  # km/s to match H(z)


# =============================================================================
# 1) DESI DR2 Table IV data 
# =============================================================================

DESI_TABLE_IV = [
    # tracer, z_eff,  DV,    sDV,    DM,     sDM,    DH,     sDH,    r_MH
    ("BGS",       0.295,  7.942, 0.075,  np.nan, np.nan, np.nan, np.nan, np.nan),
    ("LRG1",      0.510, 12.720, 0.099, 13.588, 0.167, 21.863, 0.425, -0.459),
    ("LRG2",      0.706, 16.050, 0.110, 17.351, 0.177, 19.455, 0.330, -0.404),
    ("LRG3+ELG1", 0.934, 19.721, 0.091, 21.576, 0.152, 17.641, 0.193, -0.416),
    ("ELG2",      1.321, 24.252, 0.174, 27.601, 0.318, 14.176, 0.221, -0.434),
    ("QSO",       1.484, 26.055, 0.398, 30.512, 0.760, 12.817, 0.516, -0.500),
    ("Lya",       2.330, 31.267, 0.256, 38.988, 0.531,  8.632, 0.101, -0.431),
]


@dataclass
class BaoData:
    z: np.ndarray
    d: np.ndarray
    C: np.ndarray
    labels: list


def build_desi_table_iv_dataset() -> BaoData:
    labels = []
    d_list = []
    blocks = []

    # 1) BGS isotropic DV
    for tracer, z, DV, sDV, DM, sDM, DH, sDH, rMH in DESI_TABLE_IV:
        if tracer == "BGS":
            d_list.append(DV)
            labels.append(f"{tracer}: DV/rd @ z={z}")
            blocks.append(np.array([[sDV**2]]))
            break

    # 2) anisotropic bins
    z_aniso = []
    for tracer, z, DV, sDV, DM, sDM, DH, sDH, rMH in DESI_TABLE_IV:
        if tracer == "BGS":
            continue
        z_aniso.append(z)
        d_list += [DM, DH]
        labels += [f"{tracer}: DM/rd @ z={z}", f"{tracer}: DH/rd @ z={z}"]
        cov = np.array([
            [sDM**2,         rMH*sDM*sDH],
            [rMH*sDM*sDH,    sDH**2     ]
        ])
        blocks.append(cov)

    dim = sum(b.shape[0] for b in blocks)
    C = np.zeros((dim, dim))
    i = 0
    for b in blocks:
        n = b.shape[0]
        C[i:i+n, i:i+n] = b
        i += n

    return BaoData(
        z=np.array(z_aniso, dtype=float),
        d=np.array(d_list, dtype=float),
        C=C,
        labels=labels
    )


# =============================================================================
# 1.5) CMB COMPRESSED LIKELIHOOD DATA
# =============================================================================

CMB_TABLE = [
    # R, La, ombh2, sR, sLa, sombh2, rRL, rLo, rRo
    (1.7493, 301.462, 0.02239, 0.00465, 0.0895, 0.00015, 0.47, -0.34, -0.66)
]


@dataclass
class CmbData:
    d: np.ndarray
    C: np.ndarray
    labels: list


def build_cmb_dataset() -> CmbData:
    for R, La, ombh2, sR, sLa, sombh2, rRL, rLo, rRo in CMB_TABLE:
        d_list = [R, La, ombh2]
        labels = ["R (shift parameter)", "La (acoustic scale)", "Ωb h²"]

        cov = np.array([
            [sR**2,              rRL*sR*sLa,        rRo*sR*sombh2    ],
            [rRL*sR*sLa,         sLa**2,            rLo*sLa*sombh2   ],
            [rRo*sR*sombh2,      rLo*sLa*sombh2,    sombh2**2        ]
        ])

    return CmbData(
        d=np.array(d_list, dtype=float),
        C=cov,
        labels=labels
    )


# =============================================================================
# 1.6) DES-SN5YR (DOVEKIE) SUPERNOVA DATA
# =============================================================================

@dataclass
class SnData:
    zCMB: np.ndarray
    zHEL: np.ndarray
    mu_obs: np.ndarray
    inv_cov: np.ndarray
    labels: list


def build_sn_dataset(data_file: str, covmat_file: str) -> SnData:
    
    # Parse SNANA format: lines starting with "VARNAMES:" define columns,
    # lines starting with "SN:" contain data, everything else is comment/header
    varnames = None
    rows = []
    with open(data_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('VARNAMES:'):
                varnames = line.replace('VARNAMES:', '').split()
            elif line.startswith('SN:'):
                vals = line.replace('SN:', '').split()
                rows.append(vals)

    if varnames is None:
        raise ValueError(f"No VARNAMES line found in {data_file}")

    print(f"SNANA columns: {varnames}")

    # Build arrays - first column (CID) is string, rest are numeric
    n_cols = len(varnames)
    n_rows = len(rows)

    # Find column indices
    idx_zHD = varnames.index('zHD')
    idx_zHEL = varnames.index('zHEL')
    idx_MU = varnames.index('MU')
    idx_MUERR = varnames.index('MUERR')

    zHD_all = np.array([float(r[idx_zHD]) for r in rows])
    zHEL_all = np.array([float(r[idx_zHEL]) for r in rows])
    mu_all = np.array([float(r[idx_MU]) for r in rows])

    ww = (zHD_all > 0.00)
    zCMB = zHD_all[ww]
    zHEL = zHEL_all[ww]
    mu_obs = mu_all[ww]

    print(f"Loaded {len(zCMB)} DES-SN5YR supernovae")

    # Read inverse covariance (stored as upper triangular in .npz)
    # Keys: 'nsn' = number of SN, 'cov' = upper triangular inverse covariance
    d = np.load(covmat_file)

    # Handle both key naming conventions
    if 'nsn' in d:
        n = int(d['nsn'][0])
        cov_data = d['cov']
    else:
        n = int(d[d.files[0]][0])
        cov_data = d[d.files[1]]

    inv_cov_full = np.zeros((n, n))
    inv_cov_full[np.triu_indices(n)] = cov_data

    # Reflect to lower triangular to make symmetric
    i_lower = np.tril_indices(n, -1)
    inv_cov_full[i_lower] = inv_cov_full.T[i_lower]

    # Apply the redshift cut mask
    inv_cov = inv_cov_full[ww][:, ww]

    labels = [f"SN @ z={z:.4f}" for z in zCMB]

    return SnData(
        zCMB=zCMB,
        zHEL=zHEL,
        mu_obs=mu_obs,
        inv_cov=inv_cov,
        labels=labels
    )


# =============================================================================
# 2) CAMB theory
# =============================================================================

def camb_background_and_rd(H0: float, ombh2: float, omch2: float, w0: float, wa: float,
                           tau: float = 0.054, As: float = 2.1e-9, ns: float = 0.965):

    try:
        pars = camb.CAMBparams()
        pars.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, tau=tau)
        pars.InitPower.set_params(As=As, ns=ns)
        pars.set_dark_energy(w=w0, wa=wa, dark_energy_model="ppf")
        pars.set_for_lmax(100, lens_potential_accuracy=0)

        results = camb.get_results(pars)
        rd = results.get_derived_params()["rdrag"]

        if not np.isfinite(rd) or rd <= 0:
            raise ValueError(f"Invalid rd={rd}")

        return results, rd, pars

    except Exception as e:
        raise ValueError(f"CAMB failed: {e}")


def bao_predictions(results, rd: float, z: float) -> Tuple[float, float, float]:

    chi = results.comoving_radial_distance(z)
    Hz = results.hubble_parameter(z)

    DM = chi
    DH = C_LIGHT / Hz
    DV = (DM*DM * (z*DH))**(1.0/3.0)

    return DM/rd, DH/rd, DV/rd


def cmb_predictions(results, pars, H0: float, ombh2: float) -> Tuple[float, float, float]:

    derived = results.get_derived_params()
    zstar = derived["zstar"]
    rstar = derived["rstar"]

    DA = results.angular_diameter_distance(zstar)

    try:
        Om = results.get_Omega('matter')
    except:
        Om = results.get_Omega('cdm') + results.get_Omega('baryon')

    hubble_distance = C_LIGHT / H0
    R = np.sqrt(Om) * (1.0 + zstar) * DA / hubble_distance
    La = np.pi * (1.0 + zstar) * DA / rstar

    return R, La, ombh2


def sn_distance_modulus(results, zCMB: np.ndarray, zHEL: np.ndarray) -> np.ndarray:
    """
    Compute theoretical distance modulus for SN at given redshifts.
    Using interpolation for efficiency with large SN samples.

    """
    import scipy.interpolate

    # Build a fine redshift grid for interpolation
    z_min = max(1e-4, zCMB.min() * 0.9)
    z_max = zCMB.max() * 1.1
    z_grid = np.linspace(z_min, z_max, 500)

    # Angular diameter distance on the grid
    DA_grid = np.array([results.angular_diameter_distance(z) for z in z_grid])

    f_DA = scipy.interpolate.interp1d(z_grid, DA_grid, kind='cubic')

    DA_at_data = f_DA(zCMB)

    # Distance modulus: mu = 5 * log10( (1+zCMB) * (1+zHEL) * D_A ) + 25
    mu_theory = 5.0 * np.log10((1.0 + zCMB) * (1.0 + zHEL) * np.atleast_1d(DA_at_data)) + 25.0

    return mu_theory


# =============================================================================
# 3) SN Likelihood with analytic M marginalization
# =============================================================================

def sn_log_likelihood(mu_model: np.ndarray, mu_obs: np.ndarray, inv_cov: np.ndarray) -> float:
    """
    Computes SN likelihood with analytic marginalization over absolute magnitude M.
    From Goliath et al. (2001)
    """
    delta = mu_model - mu_obs
    chit2 = delta @ inv_cov @ delta
    B = np.sum(inv_cov @ delta)
    C = np.sum(inv_cov)
    chi2 = chit2 - (B**2 / C) + np.log(C / (2 * np.pi))
    return -0.5 * chi2


# =============================================================================
# 4) Likelihoods
# =============================================================================

# Cacheing inverse covariance to avoid repeated calculations
_invC_cache = {}


def loglike_bao(theta: np.ndarray, data: BaoData) -> float:

    H0, ombh2, omch2, w0, wa = theta

    # Soft bounds
    if not (40.0 < H0 < 95.0): return -np.inf
    if not (0.015 < ombh2 < 0.03): return -np.inf
    if not (0.05 < omch2 < 0.25): return -np.inf
    if not (-3.0 < w0 < 0.0): return -np.inf
    if not (-5.0 < wa < 5.0): return -np.inf

    if 'invC_bao' not in _invC_cache:
        _invC_cache['invC_bao'] = la.inv(data.C)
    invC = _invC_cache['invC_bao']

    try:
        results, rd, pars = camb_background_and_rd(H0, ombh2, omch2, w0, wa)
    except ValueError:
        return -np.inf

    m = np.zeros_like(data.d)

    z_bgs = 0.295
    try:
        DMrd, DHrd, DVrd = bao_predictions(results, rd, z_bgs)
        if not np.isfinite(DVrd): return -np.inf
        m[0] = DVrd
    except Exception:
        return -np.inf

    idx = 1
    for zi in data.z:
        try:
            DMrd, DHrd, DVrd = bao_predictions(results, rd, zi)
            if not (np.isfinite(DMrd) and np.isfinite(DHrd)): return -np.inf
            m[idx] = DMrd
            m[idx+1] = DHrd
            idx += 2
        except Exception:
            return -np.inf

    r = data.d - m
    chi2 = r @ invC @ r

    if not np.isfinite(chi2): return -np.inf
    return -0.5 * chi2


def loglike_cmb(theta: np.ndarray, data: CmbData) -> float:

    H0, ombh2, omch2, w0, wa = theta

    if not (40.0 < H0 < 95.0): return -np.inf
    if not (0.015 < ombh2 < 0.03): return -np.inf
    if not (0.05 < omch2 < 0.25): return -np.inf
    if not (-3.0 < w0 < 0.0): return -np.inf
    if not (-5.0 < wa < 5.0): return -np.inf

    if 'invC_cmb' not in _invC_cache:
        _invC_cache['invC_cmb'] = la.inv(data.C)
    invC = _invC_cache['invC_cmb']

    try:
        results, rd, pars = camb_background_and_rd(H0, ombh2, omch2, w0, wa)
    except ValueError:
        return -np.inf

    try:
        R, La, ombh2_pred = cmb_predictions(results, pars, H0, ombh2)
        if not (np.isfinite(R) and np.isfinite(La) and np.isfinite(ombh2_pred)):
            return -np.inf
        m = np.array([R, La, ombh2_pred])
    except Exception:
        return -np.inf

    r = data.d - m
    chi2 = r @ invC @ r

    if not np.isfinite(chi2): return -np.inf
    return -0.5 * chi2


def loglike_sn(theta: np.ndarray, data: SnData) -> float:

    H0, ombh2, omch2, w0, wa = theta

    if not (40.0 < H0 < 95.0): return -np.inf
    if not (0.015 < ombh2 < 0.03): return -np.inf
    if not (0.05 < omch2 < 0.25): return -np.inf
    if not (-3.0 < w0 < 0.0): return -np.inf
    if not (-5.0 < wa < 5.0): return -np.inf

    try:
        results, rd, pars = camb_background_and_rd(H0, ombh2, omch2, w0, wa)
    except ValueError:
        return -np.inf

    try:
        mu_theory = sn_distance_modulus(results, data.zCMB, data.zHEL)
        if not np.all(np.isfinite(mu_theory)):
            return -np.inf
    except Exception:
        return -np.inf

    like = sn_log_likelihood(mu_theory, data.mu_obs, data.inv_cov)

    if not np.isfinite(like): return -np.inf
    return like


def loglike_joint(theta: np.ndarray, bao_data: BaoData, cmb_data: CmbData,
                  sn_data: Optional[SnData] = None) -> float:
    """Combined likelihood. Supports BAO+CMB, BAO+SN, CMB+SN, or BAO+CMB+SN."""
    ll = 0.0

    if bao_data is not None:
        ll_bao = loglike_bao(theta, bao_data)
        if not np.isfinite(ll_bao): return -np.inf
        ll += ll_bao

    if cmb_data is not None:
        ll_cmb = loglike_cmb(theta, cmb_data)
        if not np.isfinite(ll_cmb): return -np.inf
        ll += ll_cmb

    if sn_data is not None:
        ll_sn = loglike_sn(theta, sn_data)
        if not np.isfinite(ll_sn): return -np.inf
        ll += ll_sn

    return ll


# =============================================================================
# 5) Priors
# =============================================================================

def logprior(theta: np.ndarray, use_bbn_prior: bool = True) -> float:

    H0, ombh2, omch2, w0, wa = theta
    lp = 0.0

    # CMB directly constrains ombh2 - we use a Gaussian prior for simulations not involving CMB 
    if use_bbn_prior:
        ombh2_mu, ombh2_sig = 0.02218, 0.00055
        lp += -0.5 * ((ombh2 - ombh2_mu)/ombh2_sig)**2

    omch2_mu, omch2_sig = 0.12, 0.01
    lp += -0.5 * ((omch2 - omch2_mu)/omch2_sig)**2

    return lp


def logpost(theta: np.ndarray, data, mode: str = "bao_only") -> float:

    H0, ombh2, omch2, w0, wa = theta

    if (w0 + wa) >= 0.0:
        return -np.inf

    # Use BBN prior unless CMB is included (CMB measures ombh2 directly)
    use_bbn = mode not in ["cmb_only", "joint", "bao_cmb", "bao_cmb_sn", "cmb_sn"]
    lp = logprior(theta, use_bbn_prior=use_bbn)

    if not np.isfinite(lp):
        return -np.inf

    if mode == "bao_only":
        ll = loglike_bao(theta, data)
    elif mode == "cmb_only":
        ll = loglike_cmb(theta, data)
    elif mode == "sn_only":
        ll = loglike_sn(theta, data)
    elif mode == "bao_cmb":
        bao_data, cmb_data = data
        ll = loglike_joint(theta, bao_data, cmb_data, None)
    elif mode == "bao_sn":
        bao_data, sn_data = data
        ll = loglike_joint(theta, bao_data, None, sn_data)
    elif mode == "cmb_sn":
        cmb_data, sn_data = data
        ll = loglike_joint(theta, None, cmb_data, sn_data)
    elif mode in ["joint", "bao_cmb_sn"]:
        bao_data, cmb_data, sn_data = data
        ll = loglike_joint(theta, bao_data, cmb_data, sn_data)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return lp + ll


def prior_transform(u: np.ndarray) -> np.ndarray:

    H0    = 50.0 + 35.0*u[0]
    ombh2 = 0.020 + 0.004*u[1]
    omch2 = 0.08  + 0.10*u[2]
    w0    = -2.0 + 1.7*u[3]
    wa    = -3.0 + 6.0*u[4]
    return np.array([H0, ombh2, omch2, w0, wa])


# =============================================================================
# 6) Nautilus Sampler
# =============================================================================

# Global variables for multiprocessing
_global_data = None
_global_mode = None


def _logpost_worker(theta):
    """Top-level function for multiprocessing pool (picklable)."""
    return logpost(theta, _global_data, _global_mode)


def run_nautilus(data, n_live: int = 600, verbose: bool = True, mode: str = "bao_only",
                 pool: int = 4):

    global _global_data, _global_mode
    _global_data = data
    _global_mode = mode

    mode_names = {
        "bao_only": "BAO Only",
        "cmb_only": "CMB Only",
        "sn_only": "SN Only",
        "bao_cmb": "BAO + CMB",
        "bao_sn": "BAO + SN",
        "cmb_sn": "CMB + SN",
        "joint": "BAO + CMB + SN Joint",
        "bao_cmb_sn": "BAO + CMB + SN Joint",
    }

    print(f"\n{'='*70}")
    print(f"Starting Nautilus sampler: {mode_names.get(mode, mode)}")
    print(f"  Live points: {n_live}")
    print(f"  Pool size: {pool}")
    print(f"{'='*70}\n")

    # DIAGNOSTIC
    print("="*70)
    print("RUNNING DIAGNOSTIC TESTS")
    print("="*70)

    test_point = np.array([0.4, 0.5, 0.4, 0.45, 0.5])
    theta = prior_transform(test_point)

    print(f"\nTest Point: H0={theta[0]:.2f}, Ωbh²={theta[1]:.5f}, "
          f"Ωch²={theta[2]:.3f}, w0={theta[3]:.3f}, wa={theta[4]:.3f}")

    try:
        results_camb, rd, pars = camb_background_and_rd(
            theta[0], theta[1], theta[2], theta[3], theta[4])
        print(f"✓ CAMB: rd = {rd:.3f} Mpc")

        # BAO diagnostic
        if mode in ["bao_only", "bao_cmb", "bao_sn", "joint", "bao_cmb_sn"]:
            DMrd, DHrd, DVrd = bao_predictions(results_camb, rd, 0.295)
            if mode == "bao_only":
                data_val = data.d[0]
            elif mode in ["bao_cmb", "bao_sn"]:
                data_val = data[0].d[0]
            else:
                data_val = data[0].d[0]
            print(f"✓ BAO: DV/rd = {DVrd:.3f} (data: {data_val:.3f})")

        # CMB diagnostic
        if mode in ["cmb_only", "bao_cmb", "cmb_sn", "joint", "bao_cmb_sn"]:
            R, La, ombh2_pred = cmb_predictions(results_camb, pars, theta[0], theta[1])
            if mode == "cmb_only":
                cmb_d = data
            elif mode == "bao_cmb":
                cmb_d = data[1]
            elif mode == "cmb_sn":
                cmb_d = data[0]
            else:
                cmb_d = data[1]
            print(f"✓ CMB: R={R:.3f} (data: {cmb_d.d[0]:.3f}), "
                  f"La={La:.2f} (data: {cmb_d.d[1]:.2f})")

        # SN diagnostic
        if mode in ["sn_only", "bao_sn", "cmb_sn", "joint", "bao_cmb_sn"]:
            if mode == "sn_only":
                sn_d = data
            elif mode == "bao_sn":
                sn_d = data[1]
            elif mode == "cmb_sn":
                sn_d = data[1]
            else:
                sn_d = data[2]
            # Quick check on a few SN
            mu_test = sn_distance_modulus(results_camb, sn_d.zCMB[:5], sn_d.zHEL[:5])
            print(f"✓ SN: mu_theory[0]={mu_test[0]:.3f} (data: {sn_d.mu_obs[0]:.3f}), "
                  f"N_SN={len(sn_d.zCMB)}")

    except Exception as e:
        print(f"✗ Error: {e}")
        raise RuntimeError("Diagnostic failed")

    ll_post = logpost(theta, data, mode=mode)
    print(f"\nLog-posterior = {ll_post:.3f}")

    if not np.isfinite(ll_post):
        raise RuntimeError("Test point rejected - cannot sample!")

    if ll_post < -10000:
        print(f"\n  WARNING: Chi² ≈ {-2*ll_post:.0f} - VERY BAD FIT!")

    print("="*70 + "\n")

    sampler = Sampler(
        prior_transform,
        _logpost_worker,
        n_dim=5,
        n_live=n_live,
        pool=pool,
    )

    sampler.run(verbose=verbose)

    return sampler


# =============================================================================
# 7) Post-sampling calculations
# =============================================================================

def weighted_mean(x: np.ndarray, w: Optional[np.ndarray]) -> float:
    if w is None:
        return float(np.mean(x))
    return float(np.sum(w*x)/np.sum(w))


def weighted_cov(x: np.ndarray, y: np.ndarray, w: Optional[np.ndarray]) -> float:
    if w is None:
        xm, ym = np.mean(x), np.mean(y)
        return float(np.mean((x-xm)*(y-ym)))
    xm = weighted_mean(x, w)
    ym = weighted_mean(y, w)
    return float(np.sum(w*(x-xm)*(y-ym))/np.sum(w))


def pivot_from_samples(samples: np.ndarray, w0_idx: int, wa_idx: int,
                       weights: Optional[np.ndarray] = None) -> dict:

    w0 = samples[:, w0_idx]
    wa = samples[:, wa_idx]

    cov_w0_wa = weighted_cov(w0, wa, weights)
    var_wa    = weighted_cov(wa, wa, weights)

    if var_wa <= 0:
        raise ValueError("Variance of wa is non-positive!")

    a_p = 1.0 + cov_w0_wa / var_wa
    z_p = 1.0/a_p - 1.0

    w_p = w0 + wa*(1.0 - a_p)
    w_p_mean = weighted_mean(w_p, weights)
    w_p_sig  = np.sqrt(weighted_cov(w_p, w_p, weights))

    wa_sig = np.sqrt(var_wa)
    fom = 1.0 / (w_p_sig * wa_sig) if (w_p_sig > 0 and wa_sig > 0) else 0.0

    return {
        "a_p": float(a_p),
        "z_p": float(z_p),
        "w_p": float(w_p_mean),
        "sigma_w_p": float(w_p_sig),
        "sigma_w_a": float(wa_sig),
        "FoM_proxy": float(fom),
        "Cov_w0_wa": float(cov_w0_wa),
    }


def make_corner(samples: np.ndarray, weights: Optional[np.ndarray] = None,
                outpath: str = "corner_w0_wa.png",
                labels=None, truths=None):

    if labels is None:
        labels = [r"$H_0$", r"$\Omega_b h^2$", r"$\Omega_c h^2$", r"$w_0$", r"$w_a$"]

    fig = corner.corner(
        samples,
        weights=weights,
        labels=labels,
        show_titles=True,
        quantiles=[0.16, 0.5, 0.84],
        truths=truths,
        truth_color='red'
    )
    fig.savefig(outpath, dpi=200, bbox_inches='tight')
    print(f"Saved corner plot: {outpath}")
    plt.close(fig)


def print_parameter_summary(samples: np.ndarray, weights: Optional[np.ndarray] = None):

    labels = ["H0", "Ωb h²", "Ωc h²", "w0", "wa"]

    print("\n" + "="*70)
    print("PARAMETER SUMMARY (mean ± std)")
    print("="*70)

    for i, label in enumerate(labels):
        mean = weighted_mean(samples[:, i], weights)
        std = np.sqrt(weighted_cov(samples[:, i], samples[:, i], weights))
        print(f"{label:10s}: {mean:8.4f} ± {std:7.4f}")

    print("="*70 + "\n")


# =============================================================================
# 8) Main
# =============================================================================

def main(mode: str = "joint",
         sn_data_file: str = "DES-Dovekie_HD.csv",
         sn_covmat_file: str = "STAT+SYS.npz",
         n_live: int = 600,
         pool: int = 1):
    """
    Run the analysis.

    Modes:
        "bao_only"   - DESI BAO only
        "cmb_only"   - CMB compressed only
        "sn_only"    - DES-SN5YR only
        "bao_cmb"    - BAO + CMB
        "bao_sn"     - BAO + SN
        "cmb_sn"     - CMB + SN
        "joint"      - BAO + CMB + SN (default)
        "bao_cmb_sn" - same as "joint"
    """

    print("\n" + "="*70)
    print(f"DESI DR2 BAO + CMB + DES-SN5YR ANALYSIS")
    print(f"Mode: {mode.upper()}")
    print("="*70 + "\n")

    # Build datasets as needed
    bao_data = None
    cmb_data = None
    sn_data = None

    if mode in ["bao_only", "bao_cmb", "bao_sn", "joint", "bao_cmb_sn"]:
        bao_data = build_desi_table_iv_dataset()
        print("BAO Data:")
        for i, lab in enumerate(bao_data.labels[:3]):
            print(f"  {lab:30s} = {bao_data.d[i]:8.5f}")

    if mode in ["cmb_only", "bao_cmb", "cmb_sn", "joint", "bao_cmb_sn"]:
        cmb_data = build_cmb_dataset()
        print("\nCMB Data:")
        for i, lab in enumerate(cmb_data.labels):
            print(f"  {lab:30s} = {cmb_data.d[i]:8.5f}")

    if mode in ["sn_only", "bao_sn", "cmb_sn", "joint", "bao_cmb_sn"]:
        sn_data = build_sn_dataset(sn_data_file, sn_covmat_file)
        print(f"\nSN Data: {len(sn_data.zCMB)} supernovae, "
              f"z range [{sn_data.zCMB.min():.4f}, {sn_data.zCMB.max():.4f}]")

    # Package data for sampler
    if mode == "bao_only":
        data = bao_data
    elif mode == "cmb_only":
        data = cmb_data
    elif mode == "sn_only":
        data = sn_data
    elif mode == "bao_cmb":
        data = (bao_data, cmb_data)
    elif mode == "bao_sn":
        data = (bao_data, sn_data)
    elif mode == "cmb_sn":
        data = (cmb_data, sn_data)
    elif mode in ["joint", "bao_cmb_sn"]:
        data = (bao_data, cmb_data, sn_data)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # Run sampler
    sampler = run_nautilus(data, n_live=n_live, verbose=True, mode=mode, pool=pool)

    # Extract posterior
    try:
        points, log_w, log_l = sampler.posterior()
        samples = points
        weights = np.exp(log_w - np.max(log_w))
        weights /= weights.sum()
    except:
        post = sampler.posterior()
        samples = post["samples"]
        weights = post.get("weights", None)
        if weights is not None:
            weights /= weights.sum()

    print(f"Extracted {len(samples)} posterior samples")

    print_parameter_summary(samples, weights)

    piv = pivot_from_samples(samples, w0_idx=3, wa_idx=4, weights=weights)

    print("="*70)
    print("PIVOT REDSHIFT RESULTS")
    print("="*70)
    print(f"  Pivot redshift (z_p):             {piv['z_p']:.4f}")
    print(f"  Dark energy at pivot (w_p):       {piv['w_p']:.4f} ± {piv['sigma_w_p']:.4f}")
    print(f"  Figure of Merit (proxy):          {piv['FoM_proxy']:.2f}")
    print("="*70 + "\n")

    outfile = f"corner_posterior_{mode}.png"
    make_corner(samples, weights=weights, outpath=outfile)

    return sampler, samples, weights, piv


if __name__ == "__main__":
    warnings.filterwarnings('ignore', category=RuntimeWarning)

    # =========================================================================
    # CONFIGURE HERE
    # =========================================================================

    # Path to DES-SN5YR data files (adjust paths as needed)
    SN_DATA_FILE = "DES-Dovekie_HD.csv"
    SN_COVMAT_FILE = "STAT+SYS.npz"

    # Available modes:
    #   "bao_only", "cmb_only", "sn_only",
    #   "bao_cmb", "bao_sn", "cmb_sn",
    #   "joint" (= BAO + CMB + SN)
    MODE = "cmb_only"

    sampler, samples, weights, piv = main(
        mode=MODE,
        sn_data_file=SN_DATA_FILE,
        sn_covmat_file=SN_COVMAT_FILE,
        n_live=600,
        pool=1
    )


DESI DR2 BAO + CMB + DES-SN5YR ANALYSIS
Mode: CMB_ONLY


CMB Data:
  R (shift parameter)            =  1.74930
  La (acoustic scale)            = 301.46200
  Ωb h²                          =  0.02239

Starting Nautilus sampler: CMB Only
  Live points: 600
  Pool size: 1

RUNNING DIAGNOSTIC TESTS

Test Point: H0=64.00, Ωbh²=0.02200, Ωch²=0.120, w0=-1.235, wa=0.000
✓ CAMB: rd = 147.514 Mpc
✓ CMB: R=1.789 (data: 1.749), La=309.03 (data: 301.46)

Log-posterior = -4234.612

Starting the nautilus sampler...
Please report issues at github.com/johannesulf/nautilus.
Status    | Bounds | Ellipses | Networks | Calls    | f_live | N_eff | log Z    
Finished  | 21     | 2        | 4        | 25000    | N/A    | 10048 | -11.71   
Extracted 24999 posterior samples

PARAMETER SUMMARY (mean ± std)
H0        :  73.7887 ±  7.4666
Ωb h²     :   0.0224 ±  0.0001
Ωc h²     :   0.1217 ±  0.0014
w0        :  -1.0234 ±  0.3872
wa        :  -0.8783 ±  1.2437

PIVOT REDSHIFT RESULTS
  Pivot redshift (z_p):     

Saved corner plot: corner_posterior_cmb_only.png
